# Chapter 3 — Learning an Embedding Space

**Book alignment:** Embeddings From First Principles, Chapter 3

**Notebook role:** `ARTIFACT_REPLAY` — reads the frozen experiment artifact(s) this chapter cites and re-derives the numbers quoted in the prose (assertions fail if the artifact drifts).

**Question this notebook isolates:** Build a space from co-occurrence counts alone
(PPMI → SVD). Does it recover *topical* structure (related ≫ unrelated) while completely
failing on *assertion* (negation, relation-swap, time) — the same boundary the large neural
encoder has, only cruder?

Cells 1–4 build a homemade space on a toy corpus. Cell 5 loads the committed
`wave1/ppmi-svd-relate.json` (a 100-dim PPMI-SVD space over the full RELATE corpus) and
checks the chapter's claim against it.

In [ ]:
from pathlib import Path
import json
import numpy as np

rng = np.random.default_rng(0)


def find_repo_root(start: Path) -> Path:
    for c in (start, *start.parents):
        if (c / "experiments" / "embeddings-from-first-principles" / "wave1").is_dir():
            return c
    raise RuntimeError("run from a checkout containing experiments/embeddings-from-first-principles")


ROOT = find_repo_root(Path.cwd().resolve())
EXP = ROOT / "experiments" / "embeddings-from-first-principles"


def art(wave: str, name: str) -> dict:
    return json.loads((EXP / wave / "artifacts" / name).read_text())

## 1. A co-occurrence matrix is already a crude embedding

In [ ]:
corpus = [
    "the cat drinks milk", "the dog drinks water", "the kitten drinks milk",
    "the puppy drinks water", "the cat chases the mouse", "the dog chases the cat",
]
toks = [s.split() for s in corpus]
vocab = sorted({w for s in toks for w in s if w != "the"})
idx = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

C = np.zeros((V, V))
for s in toks:
    s = [w for w in s if w != "the"]
    for i, w in enumerate(s):
        for j in range(max(0, i - 2), min(len(s), i + 3)):
            if i != j:
                C[idx[w], idx[s[j]]] += 1

print("     " + " ".join(f"{w[:5]:>6}" for w in vocab))
for w in vocab:
    print(f"{w[:5]:>5}" + " ".join(f"{C[idx[w], idx[c]]:6.0f}" for c in vocab))
assert C[idx["kitten"], idx["milk"]] > 0 and C[idx["puppy"], idx["water"]] > 0

## 2. PPMI-weight, then compress with SVD

In [ ]:
total = C.sum()
row, col = C.sum(1, keepdims=True), C.sum(0, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    pmi = np.log((C * total) / (row @ col))
ppmi = np.nan_to_num(np.maximum(pmi, 0.0))

U, S, Vt = np.linalg.svd(ppmi)
k = 4
emb = U[:, :k] * S[:k]                        # k-D word vectors from the compressed matrix
emb = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)

def cos(a, b):
    return float(emb[idx[a]] @ emb[idx[b]])

def near(w):
    c = emb @ emb[idx[w]]
    return [vocab[i] for i in np.argsort(-c) if vocab[i] != w][:3]

print("cat   ~", near("cat"))
print("milk  ~", near("milk"))
# kitten drinks MILK, puppy drinks WATER - the cleanest signal the corpus carries
assert cos("kitten", "milk") > cos("kitten", "water")
assert cos("puppy", "water") > cos("puppy", "milk")
print("\nmeaning here = the corpus's statistical structure, compressed to low rank")

## 3. The homemade space over the full RELATE corpus (committed artifact)

In [ ]:
ps = art("wave1", "ppmi-svd-relate.json")
by = {r: v["mean"] for r, v in ps["by_relation"].items()}
for r, m in sorted(by.items(), key=lambda kv: -kv[1]):
    print(f"  {r:20} {m:.3f}")

# it SEPARATES related from unrelated ...
related = np.mean([by["paraphrase"], by["topic-related"], by["entity-related"]])
assert related - by["unrelated"] > 0.35
# ... and CONFUSES contradiction with paraphrase (the gap is inverted)
assert ps["paraphrase_vs_contradiction"] < 0                       # book: -0.10
assert by["contradiction"] > by["paraphrase"]
# a bag of term vectors cannot represent a 'not', a role reversal, or a year
assert by["negation"] > 0.9 and by["relation-swap"] >= 0.99 and by["temporal-mismatch"] > 0.99
print("\nsame failure shape as the neural encoder (nb 01): aboutness yes, assertion no - only cruder")

## What we earned

An embedding is a low-rank summary of co-occurrence structure — counting and predicting are
close routes to the same thing. The homemade 100-dim RELATE space separates *related* from
*unrelated* by 0.44 and still scores a contradiction *above* its paraphrase and a negation
at cosine 0.95. Scale sharpens the compression of the easy relations; it does not change
what is compressed away.

**Notebook 04 / Chapter 4** makes the metric an explicit decision — and shows "similar" is a
joint choice of representation *and* metric.